In [9]:
import random
import numpy as np

In [10]:
def random_pure_state(D: int) -> np.ndarray:
    psi = np.random.randn(D) + 1j * np.random.randn(D)
    psi /= np.linalg.norm(psi)
    return np.outer(psi, psi.conj())

In [11]:
si = np.array([[1,0], [0,1]])
sx = np.array([[0,1] ,[1,0]])
sy = np.array([[0,-1j], [1j,0]])
sz = np.array([[1,0], [0,-1]])

pauli = [si, sx, sy, sz]

def numberToBase(n, b):
    if n == 0:
        return [0]
    digits = []
    while n:
        digits.append(int(n % b))
        n //= b
    return digits[::-1]

# stabilizer norm (density matrix -> real number)

def get_sn(rho, qubits):
    """
    Compute stabilizer norm using proper Pauli trace method.
    Fixed according to CLAUDE.md instructions.
    """
    if qubits == 1:
        
        X = np.array([[0,1],[1,0]], dtype=np.complex128)
        Y = np.array([[0,-1j],[1j,0]], dtype=np.complex128)
        Z = np.array([[1,0],[0,-1]], dtype=np.complex128)
        
        rx = float(np.trace(rho @ X).real)
        ry = float(np.trace(rho @ Y).real)
        rz = float(np.trace(rho @ Z).real)
        
        sn = abs(rx) + abs(ry) + abs(rz)
        return sn
    ## Sep.15 Update: sum of absolute values of all non-identity Pauli trace
    # manual extraction basis  
    # 枚举 其实还挺快的
    elif qubits == 2:
        I = np.array([[1,0],[0,1]], dtype=np.complex128)
        X = np.array([[0,1],[1,0]], dtype=np.complex128)
        Y = np.array([[0,-1j],[1j,0]], dtype=np.complex128)
        Z = np.array([[1,0],[0,-1]], dtype=np.complex128)
        paulis = [I, X, Y, Z]
        sn = 0.0
        # traverse all possible combinatioms 
        for i in range(4):
            for j in range(4):
                if i == 0 and j == 0:
                    continue  # skip I dot I cuz no contribution
                sig = np.kron(paulis[i], paulis[j])
                # Kronecker product
                sn += abs(np.trace(rho @ sig))
        return sn/4
    else:
        p = int(np.log2(np.size(rho[0])))
        dim = 2**p
        dim2 = 4**p
        
        a0 = 0
        for no in range(dim2):
            ntb = numberToBase(no, 4)
            op_no = np.pad(ntb, (p-len(ntb), 0), 'constant')
            op = [[1]]
            for i in range(p):
                op = np.kron(op,pauli[op_no[i]])
            a0 = np.array(a0+op)
        a0 = a0/dim
        
        aulist = []
        for no in range(dim2):
            ntb = numberToBase(no, 4)
            op_no = np.pad(ntb, (p-len(ntb), 0), 'constant')
            op = [[1]]
            for i in range(p):
                op = np.kron(op,pauli[op_no[i]])
            aulist.append( np.dot(np.dot(op, a0), np.matrix(op).getH()) )
        
        wigner = [np.trace(np.dot(aulist[n], rho))/dim for n in range(dim2)]
        return np.real(np.sum(np.absolute(wigner)-wigner)/2)

In [12]:
n_qubits = 3
D=2**n_qubits
rho = random_pure_state(D)
print("rho:", rho)
sn = get_sn(rho, qubits=3)
print("sn value:", sn)


rho: [[ 0.00131683+1.69702009e-20j -0.0038272 +7.76221723e-03j
   0.00136661-1.17230527e-02j  0.00759555-2.92413595e-03j
  -0.01103433+6.17306405e-03j  0.0021762 -2.08219567e-02j
   0.00863856+3.37918786e-03j -0.016138  +9.48826063e-03j]
 [-0.0038272 -7.76221723e-03j  0.05687851+4.37758240e-19j
  -0.07307471+2.60158709e-02j -0.03931211-3.62741863e-02j
   0.06845765+4.71019141e-02j -0.12906217+4.76884448e-02j
  -0.00518784-6.07421285e-02j  0.10283258+6.75508473e-02j]
 [ 0.00136661+1.17230527e-02j -0.07307471-2.60158709e-02j
   0.10578229+1.00733718e-19j  0.03391468+6.45844000e-02j
  -0.06640691-9.18263223e-02j  0.18762512-2.23554900e-03j
  -0.02111798+8.04113879e-02j -0.10121691-1.33820979e-01j]
 [ 0.00759555+2.92413595e-03j -0.03931211+3.62741863e-02j
   0.03391468-6.45844000e-02j  0.05030473+1.26530586e-18j
  -0.07735431+1.11038474e-02j  0.05878927-1.15269517e-01j
   0.04232383+3.86739411e-02j -0.11415414+1.88929360e-02j]
 [-0.01103433-6.17306405e-03j  0.06845765-4.71019141e-02j
  -0.